# Notebook 26 — WU-20G guarded physical-staging qualification

**CPU ONLY · NO GPU TRAINING · NO REAL PATIENT DATASET MATERIALIZATION.**

This performs an independent read-only, 149-case capacity/validity preflight and a true **synthetic** 506/507 physical staging exercise with nnU-Net 2.8.1. The production `--mode stage` is intentionally disabled. All 15 anatomical-mask reviews and the signed prospective normalization policy remain pending.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Frozen sources, policy evidence and SHA-pinned implementation

The real 149-case step reads *only* source file metadata and already verified WU-20F header checkpoints; it does not load NCCT/perfusion voxels or reference lesion label contents.


In [ ]:
import hashlib, json, os, subprocess, sys, urllib.request
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW=ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
PRE=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS=PRE/'splits_final.json'
PLANS=PRE/'nnUNetPlans.json'
FOLDS=ROOT/'results'/'wu20b_full_fold_calibration'/'fold_private'
WU20E=ROOT/'results'/'wu20e_review_and_release_gate'
GATE=WU20E/'WU20E_release_gate_aggregate.json'
POLICY=WU20E/'WU20E_policy_PROPOSED_NOT_APPROVED_PRIVATE.json'
WU20F=ROOT/'results'/'wu20f_virtual_staging_and_forward'
PREFLIGHT=WU20F/'WU20F_preflight_aggregate.json'
CHECKPOINTS=WU20F/'patient_header_checkpoints'
OUT=ROOT/'results'/'wu20g_physical_staging_qualification'
SOURCE=Path('/content/wu20g_pinned_source')
SOURCE.mkdir(parents=True,exist_ok=True)
(SOURCE/'src'/'hybridstrokeseg').mkdir(parents=True,exist_ok=True)
(SOURCE/'scripts').mkdir(parents=True,exist_ok=True)
(SOURCE/'src'/'hybridstrokeseg'/'__init__.py').write_text('')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
assert RAW.is_dir() and FOLDS.is_dir() and GATE.is_file() and POLICY.is_file()
assert PREFLIGHT.is_file() and CHECKPOINTS.is_dir()
assert sha256(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(PLANS)=='6203521136de2f57bbe3895a639863112a5869697de9b752ade42dc1a23e4dca'
COMMIT='9d670978eeb891018d438a95c53b27f6a57b64f9'
BASE=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}'
FILES=(
    'src/hybridstrokeseg/wu19_normalization_protocol.py',
    'src/hybridstrokeseg/wu20_fold_preprocessing.py',
    'scripts/preflight_isles2024_wu20f_staging.py',
    'scripts/qualify_isles2024_nnunet_runtime.py',
    'scripts/stage_isles2024_fold_pilot.py',
    'scripts/stage_isles2024_wu20g_physical.py',
)
for name in FILES:
    dest=SOURCE/name
    dest.parent.mkdir(parents=True,exist_ok=True)
    with urllib.request.urlopen(f'{BASE}/{name}',timeout=100) as response:
        dest.write_bytes(response.read())
    print('Pinned source:',name,sha256(dest))


## 2. Install exact nnU-Net 2.8.1 and NIfTI IO dependencies

Failures are fatal: do not silently upgrade to another nnU-Net release.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
                'nnunetv2==2.8.1','nibabel>=5.2,<6'],check=True)
import importlib.metadata
assert importlib.metadata.version('nnunetv2')=='2.8.1'


## 3. Verify 149 cases and estimate full five-fold storage — no patient volume reads

This produces a **blocked** staging plan. `scratch_meets_conservative_reservation` can be false for Colab; that is expected and does not fail this planning-only notebook.


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
ENV=os.environ.copy()
ENV['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+str(SOURCE/'scripts')+os.pathsep+ENV.get('PYTHONPATH','')
SCRIPT=SOURCE/'scripts'/'stage_isles2024_wu20g_physical.py'
PLAN_CMD=[sys.executable,'-u',str(SCRIPT),'--mode','plan',
    '--raw505',str(RAW),'--splits',str(SPLITS),'--plans',str(PLANS),
    '--folds',str(FOLDS),'--wu20e-gate',str(GATE),'--wu20e-policy',str(POLICY),
    '--wu20f',str(PREFLIGHT),'--header-checkpoints',str(CHECKPOINTS),
    '--scratch','/content/wu20g_real_staging_DISABLED','--output',str(OUT)]
LOG=OUT/'WU20G_CPU_qualification.log'
with LOG.open('w',encoding='utf8') as log:
    with subprocess.Popen(PLAN_CMD,env=ENV,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        rc=proc.wait()
if rc: raise RuntimeError(f'WU20G 149-case storage plan failed: exit={rc}; inspect {LOG}')
PLAN=json.loads((OUT/'WU20G_physical_staging_plan_aggregate.json').read_text())
assert PLAN['n_patients']==149 and PLAN['n_image_headers_verified']==745
assert PLAN['staging_materialized'] is False and PLAN['training_ready'] is False
print('Uncompressed image estimate GiB:',round(PLAN['storage_estimate']['uncompressed_images_bytes']/2**30,2))
print('Recommended scratch space GiB:',round(PLAN['storage_estimate']['recommended_free_bytes_at_least']/2**30,2))
print('Scratch meets reservation:',PLAN['scratch_meets_conservative_reservation'])


## 4. Exercise two **synthetic** native physical datasets and official nnU-Net preprocessing

This step builds fake `sub_stroke9001/9002` inputs only, verifies training-label isolation and paired NCCT byte hashes, and deletes all fake NIfTIs afterward. It cannot create real Dataset506/507 patient datasets.


In [ ]:
SIM_CMD=[sys.executable,'-u',str(SCRIPT),'--mode','synthetic',
    '--plans',str(PLANS),'--output',str(OUT)]
r=subprocess.run(SIM_CMD,env=ENV,text=True,capture_output=True)
print(r.stdout)
with LOG.open('a',encoding='utf8') as log:
    log.write('\n[WU20G synthetic]\n'+r.stdout+'\n'+r.stderr)
if r.returncode:
    print(r.stderr)
    raise RuntimeError(f'WU20G synthetic official nnU-Net preprocessing failed: {r.returncode}')
SIM=json.loads((OUT/'WU20G_synthetic_staging_aggregate.json').read_text())
assert SIM['status']=='SYNTHETIC_PHYSICAL_STAGING_QUALIFIED'
assert SIM['case_count']==2 and SIM['training_label_case_count']==1
assert SIM['heldout_label_opened'] is False and SIM['training_ready'] is False
assert SIM['staging_materialized_for_real_patients'] is False
for key in ('imagesTr','imagesTs'):
    assert SIM['official_nnunet_preprocessing'][key]['post_resample_ncct_max_abs_difference']<=1e-5
print('Synthetic staging passed; real patient dataset staging remains BLOCKED.')


## 5. Immutable outcome summary (clinical release still blocked)

The final JSON contains no patient images, lesion data or training commands. The original Dataset504 and Dataset505 remain frozen.


In [ ]:
RESULT={'schema':'WU20G-colab26-summary-v1',
    'patient_header_checks':PLAN['n_image_headers_verified'],
    'patients_in_plan':PLAN['n_patients'],
    'estimated_uncompressed_storage_bytes':PLAN['storage_estimate']['combined_uncompressed_bytes'],
    'synthetic_physical_staging':'PASS',
    'actual_patient_physical_staging':False,
    'authentic_clinical_approval_obtained':False,
    'real_model_training_started':False,
    'training_ready':False}
(OUT/'WU20G_qualification_summary.json').write_text(json.dumps(RESULT,indent=2)+'\n')
print(json.dumps(RESULT,indent=2))
